In [1]:
import os
import re
import random
import string
import requests

import numpy as np
import pandas as pd
import geopandas as gpd

import arcpy
import arcpy.sa as sa
import arcpy.mp as mp
arcpy.CheckOutExtension("Spatial")

import time

import shutil

### Merge spatial statistics and Census data to a new CSV:

In [2]:
print(os.path.exists("output/combined_spatial_stats.csv"))
print(os.path.exists("output/census_cbg_data.csv"))

False
True


In [4]:
# Load the two CSV files
combined_stats = pd.read_csv("output/combined_spatial_stats.csv")
census_data = pd.read_csv("output/census_cbg_data.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'output/combined_spatial_stats.csv'

In [ ]:
# Merge on GEOIDFQ
merged_df = pd.merge(
    combined_stats,
    census_data,
    left_on="GEOIDFQ",
    right_on="GEO_ID",
    how="left"
)

In [ ]:
# Save the result to a new CSV file
merged_df.to_csv("output/spatial_stats_and_census.csv", index=False)

In [ ]:
shutil.move("output/combined_spatial_stats.csv", "output/temp/combined_spatial_stats.csv")
shutil.move("output/census_cbg_data.csv", "output/temp/census_cbg_data.csv")

### Cleanup the output CSV:

In [ ]:
# Load the CSV into a DataFrame
input_path = "output/spatial_stats_and_census.csv"
workingdf = pd.read_csv(input_path)

# Move the original CSV to output/temp/
#temp_dir = "output/temp/"
#os.makedirs(temp_dir, exist_ok=True)  # Ensure the temp folder exists
#shutil.move(input_path, os.path.join(temp_dir, "census_cbg_data.csv"))

# Remove GEO_ID
#workingdf = workingdf.drop(columns=["GEOIDFQ"])

# Reorder columns
cols = workingdf.columns.tolist() #Create list of columns
#cols.remove("GEO_ID") #remove from the LIST
#cols.remove("NAME") #remove from the LIST

#choose front columns to move to beginning
front_cols = [col for col in ["GEO_ID", "NAME"] if col in cols]

# Build the remaining list, excluding front_cols
remaining_cols = [col for col in cols if col not in front_cols]

workingdf = workingdf[front_cols + remaining_cols]

# Drop unneeded fields
#workingdf = workingdf.drop(columns=["GEOIDFQ"])
#workingdf = workingdf.drop(columns=["GEO_ID2.1"])

# make list of fields to drop
pattern = re.compile(r"^GEO_ID_2(\..*)?$")
cols_to_drop = [col for col in workingdf.columns if re.match(pattern, col)]
workingdf = workingdf.drop(columns=cols_to_drop)

# drop additional fields
workingdf = workingdf.drop(columns=["GEOIDFQ"])

# Check results
workingdf.head()

### Calculate derived fields here:

In [ ]:
workingdf['LimitedEnglishHH'] = workingdf['B16002_004E'] + workingdf['B16002_007E'] + workingdf['B16002_010E'] + workingdf['B16002_013E'] + workingdf['B16002_016E'] + workingdf['B16002_019E'] + workingdf['B16002_022E'] + workingdf['B16002_025E'] + workingdf['B16002_028E'] + workingdf['B16002_031E'] + workingdf['B16002_034E'] + workingdf['B16002_037E']

In [ ]:
# Write the DataFrame to a new CSV file
output_path = "output/cbg_demog_enviro_calc.csv"
workingdf.to_csv(output_path, index=False)

print("Output files created at: " + output_folder_path)